# Final Free Local LLM Evidence Review

This notebook uses Qwen locally in Google Colab to review one already-created evidence report at a time. It never installs or executes the target package. The notebook checks the embedded package name before generating or downloading a result.

In [ ]:
# Step 1: Upload ai_package_detection_project_llm_colab.zip.
from google.colab import files
project_upload = files.upload()
print('Uploaded:', list(project_upload))

In [ ]:
# Step 2: Extract the project and install the optional local-LLM packages.
!unzip -q /content/ai_package_detection_project_llm_colab.zip -d /content
%cd /content/ai_package_detection_project
!pip -q install -r requirements-colab-llm.txt
!nvidia-smi || true

In [ ]:
# Step 3: Upload ONE evidence report. Choose either the requests or lodash report.
from pathlib import Path
import json

evidence_upload = files.upload()
uploaded_evidence_name = next(iter(evidence_upload))
EVIDENCE = str(Path.cwd() / uploaded_evidence_name)
evidence = json.loads(Path(EVIDENCE).read_text(encoding='utf-8'))
PACKAGE_NAME = evidence['package']['package_name']
print('Evidence file:', EVIDENCE)
print('Uploaded package:', PACKAGE_NAME)
print('Triage action:', evidence['decision_agent']['decision'])

In [ ]:
# Step 4: Select the output filename only after confirming the package printed above.
# For requests-2.32.3 use requests_llm_review.json. For lodash-4.17.21 use lodash_llm_review.json.
OUTPUT = '/content/requests_llm_review.json' if PACKAGE_NAME == 'requests-2.32.3' else '/content/lodash_llm_review.json'
print('Output file:', OUTPUT)
!python run_local_llm_review.py --evidence $EVIDENCE --output $OUTPUT --max-new-tokens 350

In [ ]:
# Step 5: Verify that the LLM result belongs to the same uploaded package, then download it.
review = json.loads(Path(OUTPUT).read_text(encoding='utf-8'))
review_package = review['bounded_evidence']['package']['package_name']
assert review_package == PACKAGE_NAME, f'Package mismatch: {review_package} != {PACKAGE_NAME}'
print('Verified result package:', review_package)
print(json.dumps(review['structured_review'], indent=2))
files.download(OUTPUT)